# Phase 4: Model Training & Evaluation
This file contains ONLY the models. It expects `X_train`, `X_test`, `y_train`, `y_test` to be prepared.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, f1_score

# --- IMPORTING TREE-BASED MODELS ---
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from xgboost import XGBClassifier

import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 1. CUSTOM LOGISTIC REGRESSION (FROM SCRATCH)
# ==========================================
class CustomLogisticRegression:
    """
    Logistic Regression built entirely from scratch using Numpy and Gradient Descent.
    """
    def __init__(self, learning_rate=0.01, num_iterations=1000):
        self.lr = learning_rate
        self.iterations = num_iterations
        self.weights = None
        self.bias = None
        
    def _sigmoid(self, z):
        # Clip z to prevent overflow in exp
        z = np.clip(z, -250, 250)
        return 1 / (1 + np.exp(-z))
    
    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0
        
        for _ in range(self.iterations):
            linear_model = np.dot(X, self.weights) + self.bias
            y_pred = self._sigmoid(linear_model)
            
            # Calculate gradients
            dw = (1 / n_samples) * np.dot(X.T, (y_pred - y))
            db = (1 / n_samples) * np.sum(y_pred - y)
            
            # Update weights & bias
            self.weights -= self.lr * dw
            self.bias -= self.lr * db

    def predict_proba(self, X):
        linear_model = np.dot(X, self.weights) + self.bias
        return self._sigmoid(linear_model)
    
    def predict(self, X, threshold=0.5):
        probs = self.predict_proba(X)
        return np.array([1 if p >= threshold else 0 for p in probs])

# ==========================================
# 2. MODEL CONFIGURATIONS & PIPELINE
# ==========================================

def get_all_models():
    """
    Returns a dictionary of all initialized models with their full configurations.
    """
    models = {
        "Custom Logistic Regression": CustomLogisticRegression(
            learning_rate=0.05, 
            num_iterations=2000
        ),
        
        "Random Forest": RandomForestClassifier(
            n_estimators=200,          # Number of trees
            max_depth=10,              # Limit depth to prevent overfitting
            min_samples_split=5,       # Minimum samples required to split a node
            class_weight='balanced',   # Handle imbalance internally if not using SMOTE
            random_state=42
        ),
        
        "XGBoost": XGBClassifier(
            n_estimators=200,          # Number of boosting rounds
            learning_rate=0.05,        # Step size shrinkage
            max_depth=5,               # Depth of each tree
            subsample=0.8,             # Use 80% of data per tree (prevents overfitting)
            colsample_bytree=0.8,      # Use 80% of features per tree
            eval_metric='logloss',
            random_state=42
        ),
        
        "AdaBoost": AdaBoostClassifier(
            n_estimators=100,          # Number of weak learners
            learning_rate=0.1,         # Weight applied to each classifier
            random_state=42
        )
    }
    return models

# ==========================================
# 3. TRAINING AND EVALUATION EXECUTION
# ==========================================

def train_and_evaluate(X_train, X_test, y_train, y_test):
    """
    Loops through all models, trains them, and evaluates their performance.
    """
    models = get_all_models()
    results_list = []
    
    print("="*50)
    print("STARTING MODEL TRAINING & EVALUATION PIPELINE")
    print("="*50)

    for name, model in models.items():
        print(f"\nTraining [{name}]...")
        
        # 1. Train the model
        model.fit(X_train, y_train)
        
        # 2. Make predictions
        y_pred = model.predict(X_test)
        
        # Determine how to get probabilities depending on model type
        if hasattr(model, "predict_proba"):
            y_prob = model.predict_proba(X_test)
            # Handle sklearn models returning 2D arrays vs custom returning 1D
            if len(y_prob.shape) == 2:
                y_prob = y_prob[:, 1]
        else:
            y_prob = y_pred # Fallback

        # 3. Calculate metrics
        accuracy = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        roc_auc = roc_auc_score(y_test, y_prob)
        
        # 4. Save results
        results_list.append({
            "Model": name,
            "Accuracy": accuracy,
            "F1-Score": f1,
            "ROC-AUC": roc_auc
        })
        
        print(f"--> Done! ROC-AUC: {roc_auc:.4f}")

    # Display final leaderboard
    print("\n" + "="*50)
    print("FINAL LEADERBOARD")
    print("="*50)
    results_df = pd.DataFrame(results_list)
    results_df = results_df.sort_values(by="ROC-AUC", ascending=False).reset_index(drop=True)
    print(results_df.to_string(index=False))
    
    return models, results_df

### How to run this file:
Uncomment the code below and pass your actual SMOTE-augmented data to run the pipeline.

In [ ]:
if __name__ == "__main__":
    # --- DUMMY DATA FOR SCRIPT TESTING ---
    # Replace this block with importing your actual X_train_smote, X_test, y_train_smote, y_test
    # Example: 
    # from data_generator import X_train_smote, X_test, y_train_smote, y_test
    
    print("Generating temporary dummy data just to test the script execution...")
    np.random.seed(42)
    X_dummy = np.random.rand(1000, 20) # 1000 samples, 20 features
    y_dummy = np.random.randint(0, 2, 1000)
    from sklearn.model_selection import train_test_split
    X_tr, X_te, y_tr, y_te = train_test_split(X_dummy, y_dummy, test_size=0.2, random_state=42)
    
    # --- RUN THE PIPELINE ---
    trained_models, leaderboard = train_and_evaluate(X_tr, X_te, y_tr, y_te)